In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../../'))
sys.path.append(base_path)

import tensorflow as tf
from tensorflow.keras.callbacks import TensorBoard

import numpy as np
from tqdm import tqdm
import datetime

from data_loaders.data_loader_complete import DataLoaderComplete
import project_images_loader as pil
import matplotlib.pyplot as plt

from discriminator import MetamorphicDiscriminator
from generator import MetamorphicGenerator

import utils.functions as fc
import utils.visualization as vis
import utils.util_deep_learning as udl
import utils.util as util

# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

2024-04-28 14:59:31.504529: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-28 14:59:31.648291: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-04-28 14:59:31.682763: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-04-28 14:59:32.606296: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; 

SERVER: True




TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.13.0 and strictly below 2.16.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.10.1 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're using a tested and supported configuration, either change the TensorFlow version or the TensorFlow Addons's version. 
You can find the compatibility matrix in TensorFlow Addon's readme:
https://github.com/tensorflow/

## Visualize data

In [2]:

# %matplotlib widget

# data_train = DataLoader(batch_size=3, image_size=(193,229,193), shuffle=False, channels_first=False, image_sufix=pil.INAME_PREPROCESSED, nb_patients=-1, server=False)
# # fc.imgshow_3D("", data_train[0][0][0,0])
# fc.imgshow_3D("", data_train[0][0][0,...,0])

### Validation visualization

In [3]:
def show_generated_image(id, G, img_size, current_plane=116):
    def fast_resieze(img, img_size):
        return img[:img_size[0], :img_size[1], :img_size[2]]
    
    img_hunt3 = fast_resieze(pil.load_preprocessed_image(3, id, absolute=True)[0], img_size)
    img_hunt4 = fast_resieze(pil.load_preprocessed_image(4, id, absolute=True)[0], img_size)

    img_gen_h4 = G(np.expand_dims(np.expand_dims(img_hunt3, axis=0), axis=-1), training=False)
    img_gen_h4 = img_gen_h4[-1][0, ...,0].numpy()
    img_gen_h4[img_gen_h4 < 0] = 0
    img_gen_h4 = img_gen_h4 / np.max(img_gen_h4)
    vis.shows_3_images(img_hunt3, img_hunt4, img_gen_h4, current_plane=current_plane)



### Loss functions

In [4]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(disc_output_real1, disc_output_real2, disc_output_real3,
                        disc_output_fake1, disc_output_fake2, disc_output_fake3):
    
    real_loss1 = cross_entropy(tf.ones_like(disc_output_real1), disc_output_real1)
    real_loss2 = cross_entropy(tf.ones_like(disc_output_real2), disc_output_real2)
    real_loss3 = cross_entropy(tf.ones_like(disc_output_real3), disc_output_real3)
    
    fake_loss1 = cross_entropy(tf.zeros_like(disc_output_fake1), disc_output_fake1)
    fake_loss2 = cross_entropy(tf.zeros_like(disc_output_fake2), disc_output_fake2)
    fake_loss3 = cross_entropy(tf.zeros_like(disc_output_fake3), disc_output_fake3)

    # total_loss = ((real_loss1 + real_loss2 + real_loss3)/3 + (fake_loss1 + fake_loss2 + fake_loss3))/2

    real_loss = real_loss1 + real_loss2 + real_loss3
    fake_loss = fake_loss1 + fake_loss2 + fake_loss3
    total_loss = real_loss + fake_loss
    return total_loss, real_loss, fake_loss

def generator_loss(disc_output_fake1, disc_output_fake2, disc_output_fake3, y_batch, 
                    y_batch_fake1, y_batch_fake2, y_batch_fake3):
    
    gan_loss1 = cross_entropy(tf.ones_like(disc_output_fake1), disc_output_fake1)
    gan_loss2 = cross_entropy(tf.ones_like(disc_output_fake2), disc_output_fake2)
    gan_loss3 = cross_entropy(tf.ones_like(disc_output_fake3), disc_output_fake3)

    y_batch_2 = tf.nn.pool(y_batch, window_shape=(2, 2, 2), pooling_type='AVG', strides=(2, 2, 2), padding='VALID')
    y_batch_1 = tf.nn.pool(y_batch_2, window_shape=(2, 2, 2), pooling_type='AVG', strides=(2, 2, 2), padding='VALID')

    l1_loss1 = tf.reduce_mean(tf.abs(y_batch_1 - y_batch_fake1))
    l1_loss2 = tf.reduce_mean(tf.abs(y_batch_2 - y_batch_fake2))
    l1_loss3 = tf.reduce_mean(tf.abs(y_batch - y_batch_fake3))

    l1_loss = l1_loss1 + l1_loss2 + l1_loss3
    gan_loss = gan_loss1 + gan_loss2 + gan_loss3
    
    total_loss = gan_loss + 25.0 * l1_loss

    return total_loss, gan_loss, l1_loss


In [5]:
@tf.function
def train_step(G, D, x_batch, y_batch, g_optimizer, d_optimizer):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        y_batch_fake1, y_batch_fake2, y_batch_fake3  = G(x_batch, training=True)

        disc_output_real1, disc_output_real2, disc_output_real3 = D([x_batch, y_batch], training=True)
        disc_output_fake1, disc_output_fake2, disc_output_fake3 = D([x_batch, y_batch_fake3], training=True)

        disc_loss, disc_real_loss, disc_fake_loss = discriminator_loss(disc_output_real1, disc_output_real2, disc_output_real3,
                                       disc_output_fake1, disc_output_fake2, disc_output_fake3)
        
        gen_loss, gan_loss, l1_loss = generator_loss(disc_output_fake1, disc_output_fake2, disc_output_fake3, y_batch, 
                                                     y_batch_fake1, y_batch_fake2, y_batch_fake3)

    gradients_of_generator = gen_tape.gradient(gen_loss, G.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, D.trainable_variables)

    g_optimizer.apply_gradients(zip(gradients_of_generator, G.trainable_variables))
    d_optimizer.apply_gradients(zip(gradients_of_discriminator, D.trainable_variables))

    return gen_loss, disc_loss, gan_loss, l1_loss, disc_real_loss, disc_fake_loss



### Training

In [6]:

def train(num_epochs, batch_size):
    # input_shape = (193,229,193)
    input_shape = (192,224,192)
    # input_shape = (64,64,64)

    # ---- Models
    D = MetamorphicDiscriminator()
    G = MetamorphicGenerator()

    # optimizers
    inital_lr = 1e-4
    g_optimizer = tf.keras.optimizers.Adam(inital_lr)
    d_optimizer = tf.keras.optimizers.Adam(inital_lr)

    # ---- Data
    server = False
    data_train = DataLoaderComplete(image_sufix=pil.INAME_PREPROCESSED,
                                    batch_size=batch_size, 
                                    image_size=input_shape, 
                                    channels_first=False,
                                    shuffle=True, 
                                    nb_patients=300, 
                                    shuffle_before_selecting=False, 
                                    server=server)

    # visualization
    nb_loss = 6
    acum_loss_train = udl.AcumulativeMetrics(nb_metrics = nb_loss)
    # tensorboard_callback = TensorBoard(log_dir=f"{current_path}/models/logs")

    log_dir=f"{current_path}/models/logs/"
    summary_writer = tf.summary.create_file_writer(
    log_dir + "fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
    # %load_ext tensorboard
    # %tensorboard --logdir {log_dir}

    # ---- checkpoint
    checkpoint_dir = f"{current_path}/models/training_checkpoints/"
    checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
    checkpoint = tf.train.Checkpoint(generator_optimizer=g_optimizer,
                                    discriminator_optimizer=d_optimizer,
                                    generator=G,
                                    discriminator=D)
    # checkpoint_manager = tf.train.CheckpointManager(checkpoint, checkpoint_dir)
    # restore checkpoint
    checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

    for epoch in range(0,num_epochs):

        print(f"\nEpoch {epoch+1}/{num_epochs}")
        
        # visualize images
        show_generated_image("08202", G, input_shape, current_plane=116)

        # ---- Train
        acum_loss_train.reset()
        progress_bar = tqdm(total=len(data_train), desc=f'Initializing...')
        for step, (x_batch, y_batch) in enumerate(data_train):
            gen_loss, disc_loss, gan_loss, l1_loss, disc_real_loss, disc_fake_loss = train_step(G, D, x_batch, y_batch, g_optimizer, d_optimizer)
            acum_loss_train.update((gen_loss, disc_loss, gan_loss, l1_loss, disc_real_loss, disc_fake_loss))
            loss_mean = acum_loss_train.get_mean()
            
            progress_bar.set_description(desc=f'gen: {loss_mean[0]:.5f} \t disc: {loss_mean[1]:.5f} \t gan: {loss_mean[2]:.5f} \t l1: {loss_mean[3]:.5f} \t disc_real: {loss_mean[4]:.5f} \t disc_fake: {loss_mean[5]:.5f}')
            progress_bar.update(1)
        del progress_bar

        # ---- Check points
        if  epoch > 0 and epoch % 2 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)
        
        # ---- Learning rate
        # if epoch > 44 and epoch % 22 == 0:
        if epoch > 44 and epoch % 22 == 0:
            inital_lr *= 0.5
            g_optimizer.learning_rate = inital_lr
            d_optimizer.learning_rate = inital_lr

        # ---- tensorboard
        loss_mean = acum_loss_train.get_mean()
        with summary_writer.as_default():
            tf.summary.scalar('generator_loss', loss_mean[0], step=epoch)
            tf.summary.scalar('discriminator_loss', loss_mean[1], step=epoch)
            tf.summary.scalar('gan_loss', loss_mean[2], step=epoch)
            tf.summary.scalar('l1_loss', loss_mean[3], step=epoch)
            tf.summary.scalar('disc_real_loss', loss_mean[4], step=epoch)
            tf.summary.scalar('disc_fake_loss', loss_mean[4], step=epoch)
            

train(num_epochs=50, batch_size=1)

2024-04-28 14:59:36.378101: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-28 14:59:37.150653: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38206 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB, pci bus id: 0000:8a:00.0, compute capability: 8.0



Epoch 1/50


2024-04-28 14:59:38.879888: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8100
2024-04-28 14:59:39.744911: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-04-28 14:59:41.423894: I tensorflow/stream_executor/cuda/cuda_blas.cc:1614] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


In [ ]:
# Sim: 0.00000 	 gen_loss: 0.93415 	 disc_loss: 1.11709: 100%|██████████| 8/8 [00:14<00:00,  1.86s/it]
# Sim: 0.00000 	 gen_loss: 1.22538 	 disc_loss: 1.64933: 100%|██████████| 8/8 [00:14<00:00,  1.86s/it]